TEXT SUMMARIZER :
steps:
1. load dataset
2. random sampling
3. preprocessing
4. Tokenization
5. load model (T5Tokenizer)
6. fine tuning(define arguments
trainer values)
7. training model
8. save and load the model
9. testing...

In [3]:
import numpy as np
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

1. Load Dataset

In [4]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [5]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [6]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [7]:
train_data.sample(5)

,id,dialogue,summary
9522,13680440,Damarion: Wake up! Let me know when you wake u...,Damarion woke Mara up. Damarion drinks a beer ...
7020,13611572,John: hey madam Lucy\r\nLucy: hey\r\nJohn: got...,John wants to know if his CAT marks are update...
597,13865260,Garry: <photo_file>\nIdan: What is this?\nGarr...,Garry sends Olivia and Idan the recipe which O...
9271,13862584,Caleb: so what about the trip? u coming?\nEliz...,Caleb and Elizabeth are going on a trip. Ms Sm...
8491,13728930,David: I'm in the shop right now and I want to...,David will buy 2 bottles of wine for Barbara.


In [8]:
train_data.shape, val_data.shape

((14732, 3), (818, 3))

2. Ramdom Sampling

In [9]:
#random sampling(training only on few data).

train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [10]:
train_data.shape, val_data.shape

((4000, 3), (500, 3))

3. Data Preprossesing

In [11]:
import re

def clean_data(text):
  text = re.sub(r"\r\n", " ", text) #remove lines
  text = re.sub(r"\s+", " ", text) #remove spaces
  text = re.sub(r"<.*?>", " ", text)
  text.strip().lower()
  return text

In [12]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [13]:
train_data["dialogue"][0]

"Violet: hi! i came across this Austin's article and i thought that you might find it interesting Violet:   Claire: Hi! :) Thanks, but I've already read it. :) Claire: But thanks for thinking about me :)"

4. Tokenization

In [14]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [29]:
# raw data => tokenized inputs for fine-tuning

def tokenize(data):
  inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
  target = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True) #150 cuz summary smaller than dialogue

  inputs["labels"] = target["input_ids"]
  return inputs

In [30]:
# inputs["labels"] = target["input_ids"].
# This means the model will use the dialogue tokens as input and the summary tokens as the expected output (labels) during training.

In [31]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [32]:
train_dataset[0]

{'input_ids': [28866, 10, 7102, 55, 3, 23, 764, 640, 48, 8513, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 28866, 10, 19542, 10, 2018, 55, 3, 10, 61, 1333, 6, 68, 27, 31, 162, 641, 608, 34, 5, 3, 10, 61, 19542, 10, 299, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [33]:
# input ids : token ids of dialogue
# 1 => EOS, 0 => padding

# attention mask : tells which values are valid, which values are padding vaues.

# label : token ids of summary

In [34]:
len(train_dataset[0]["input_ids"])

512

In [35]:
type(train_dataset)
type(val_dataset)

list

6. Working with our model (Load Model)

In [36]:
#NLP => generation task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

7. Defining Training Argumments

In [41]:
# defining training argumments

training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs=5,
    weight_decay = 0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
    # 0 => lr default
)

In [42]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

8. Training

In [43]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.616270,0.388732
2,0.408453,0.367727
3,0.385911,0.360615
4,0.374473,0.357573
5,0.369109,0.356651


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2500, training_loss=0.43084341430664064, metrics={'train_runtime': 1004.9179, 'train_samples_per_second': 19.902, 'train_steps_per_second': 2.488, 'total_flos': 2706836029440000.0, 'train_loss': 0.43084341430664064, 'epoch': 5.0})

In [44]:
# load model => fine-tune => save model.

Save model

In [45]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model") #we have to tokenize user input.

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

load model

In [47]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model") #we have to tokenize user input.

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [50]:
def summarize_dialogue(dialogue):

  #clean
  dialogue = clean_data(dialogue)

  #tokenize
  inputs = tokenizer(
      dialogue,
      padding="max_length",
      max_length=512,
      truncation=True,
      return_tensors="pt" #returns pytorch tensors
  ).to(device)

  #generate
  model.to(device)
  targets = model.generate(
      input_ids = inputs["input_ids"],
      attention_mask = inputs["attention_mask"],
      max_length=150,
      num_beams=4, #transformer will generate 4 different types of summaries and returns the best summary.
      early_stopping=True
  )

  # token ids convert to summary => decoding.

  #decoded our output.
  summary = tokenizer.decode(targets[0], skip_special_tokens=True) # special_tokens : SEP, EOS, PADDING
  return summary

In [52]:
test_dialogue = """
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye
"""

summary = summarize_dialogue(test_dialogue)

print("Summary : ", summary)

Summary :  Amanda has Betty's number. She can't find it. Larry called her last time she was at the park together.
